# Limpieza - Código Único de Medicamentos - CUM

In [ ]:
# ============================================================
# 1. LIBRERÍAS Y RUTA DEL ARCHIVO
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import re

In [ ]:
# ============================================================
# 2. RUTA DEL ARCHIVO
# ============================================================

archivo = Path("CÓDIGO_ÚNICO_DE_MEDICAMENTOS_VIGENTES_20260604.csv")

In [ ]:
# ============================================================
# 3. CARGAR DATOS
# ============================================================

df = pd.read_csv(archivo, encoding="utf-8", low_memory=False)

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

In [ ]:
# ============================================================
# 4. FUNCIÓN DE LIMPIEZA
# ============================================================

def limpiar_texto(serie):
    """Limpia espacios, saltos de línea y textos vacíos."""
    return (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    )


def limpiar_fecha(serie):
    """
    Convierte fechas a datetime y elimina fechas sentinela como:
    1900, 2099, 3000, 9999.
    """
    s = serie.astype("string").str.slice(0, 10)
    anio = pd.to_numeric(s.str.slice(0, 4), errors="coerce")

    s = s.mask(anio <= 1900)
    s = s.mask(anio >= 2099)

    return pd.to_datetime(s, errors="coerce")


def limpiar_numero(serie):
    """Convierte cantidades a número, aceptando coma decimal."""
    return pd.to_numeric(
        serie.astype("string").str.replace(",", ".", regex=False),
        errors="coerce"
    )


def limpiar_dataset_cum(df):
    """Aplica la limpieza principal al dataset CUM."""

    df = df.copy()

    # 1. Normalizar nombres de columnas
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

    # 2. Limpiar columnas de texto
    columnas_texto = df.select_dtypes(include=["object"]).columns
    for col in columnas_texto:
        df[col] = limpiar_texto(df[col])

    # 3. Crear llave CUM
    df["cum"] = (
        df["expedientecum"].astype("string").str.strip()
        + df["consecutivocum"].astype("string").str.strip()
    )

    # 4. Limpiar fechas
    columnas_fecha = [
        "fechaexpedicion",
        "fechavencimiento",
        "fechaactivo",
        "fechainactivo"
    ]

    for col in columnas_fecha:
        if col in df.columns:
            df[col + "_limpia"] = limpiar_fecha(df[col])

    # 5. Convertir cantidades
    for col in ["cantidadcum", "cantidad"]:
        if col in df.columns:
            df[col + "_num"] = limpiar_numero(df[col])

    # 6. Validar código ATC
    if "atc" in df.columns:
        patron_atc = r"^[A-Z]\d{2}[A-Z]{2}\d{2}$"
        df["atc_valido"] = df["atc"].astype("string").str.match(patron_atc)
        df["atc_valido"] = df["atc_valido"].fillna(False)

    # 7. Banderas de calidad
    df["sin_descripcion_comercial"] = df.get("descripcioncomercial", pd.Series(index=df.index)).isna()
    df["sin_unidad_referencia"] = df.get("unidadreferencia", pd.Series(index=df.index)).isna()
    df["sin_forma_farmaceutica"] = df.get("formafarmaceutica", pd.Series(index=df.index)).isna()

    if "estadocum" in df.columns and "fechainactivo_limpia" in df.columns:
        df["cum_inactivo_sin_fecha_inactivo"] = (
            df["estadocum"].eq("Inactivo") & df["fechainactivo_limpia"].isna()
        )

    return df


df = limpiar_dataset_cum(df)

In [ ]:
# ============================================================
# 5. RESUMEN DIAGNÓSTICO
# ============================================================

resumen = pd.DataFrame({
    "columna": df.columns,
    "tipo": df.dtypes.astype(str).values,
    "nulos": df.isna().sum().values,
    "%_nulos": (df.isna().mean().values * 100).round(2),
    "valores_unicos": df.nunique(dropna=True).values
}).sort_values("%_nulos", ascending=False)

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])
print("CUM únicos:", df["cum"].nunique())
print("Duplicados exactos:", df.duplicated().sum())

resumen

In [ ]:
# ============================================================
# 6. ALERTAS
# ============================================================

alertas = {
    "filas_totales": len(df),
    "columnas_totales": df.shape[1],
    "cum_unicos": df["cum"].nunique(),
    "duplicados_exactos": int(df.duplicated().sum()),
    "atc_invalidos": int((df["atc_valido"] == False).sum()) if "atc_valido" in df.columns else None,
    "sin_descripcion_comercial": int(df["sin_descripcion_comercial"].sum()),
    "sin_unidad_referencia": int(df["sin_unidad_referencia"].sum()),
    "sin_forma_farmaceutica": int(df["sin_forma_farmaceutica"].sum()),
}

pd.DataFrame([alertas]).T.rename(columns={0: "valor"})

In [ ]:
# ============================================================
# 7. TABLAS INTERMEDIAS
# ============================================================

productos = df[[
    "expediente", "producto", "titular", "registrosanitario",
    "fechaexpedicion_limpia", "fechavencimiento_limpia", "estadoregistro"
]].drop_duplicates()

presentaciones = df[[
    "cum", "expedientecum", "consecutivocum", "cantidadcum_num",
    "unidad", "descripcioncomercial", "estadocum",
    "fechaactivo_limpia", "fechainactivo_limpia",
    "muestramedica", "formafarmaceutica"
]].drop_duplicates()

principios_activos = df[[
    "cum", "principioactivo", "concentracion",
    "cantidad_num", "unidadreferencia", "atc", "descripcionatc", "atc_valido"
]].drop_duplicates()

roles = df[[
    "cum", "nombrerol", "tiporol", "modalidad"
]].drop_duplicates()

print("Productos:", productos.shape)
print("Presentaciones:", presentaciones.shape)
print("Principios activos:", principios_activos.shape)
print("Roles:", roles.shape)

In [ ]:
# ============================================================
# 8. EXPORTAR RESULTADOS
# ============================================================

salida = Path("Tablas_CUM")
salida.mkdir(exist_ok=True)

# CSV Tablas Intemedias
df.to_csv(salida / "cum_base_limpia.csv", index=False, encoding="utf-8")
productos.to_csv(salida / "productos.csv", index=False, encoding="utf-8")
presentaciones.to_csv(salida / "presentaciones.csv", index=False, encoding="utf-8")
principios_activos.to_csv(salida / "principios_activos.csv", index=False, encoding="utf-8")
roles.to_csv(salida / "roles.csv", index=False, encoding="utf-8")